# **03 - Build `shows` table (Gametime)**
***
## What this notebook is for

**What:** Turns the Gametime price database into one clean row per show, with the artist name, venue, date features and the price about 14 days before the show.

**Why:** `music.db` has about 1.9 million rows of daily snapshots, but the model needs one row per show with a single price to predict. This notebook decides which shows count, what the artist name is, and which price is the target. Every other table (artists, venues) joins to this one.

**What it does:**
1. Reads `data/music.db` (observed days with prices only).
2. Drops festival and multi-day passes and tribute / party nights, and cleans each show name into an artist name (for example, "Kelly Clarkson (Rescheduled from 8/8/2025)" becomes "Kelly Clarkson").
3. For each show, keeps the snapshot closest to 14 days before the show and records its lowest (`price_min`) and median (`price_median`) price, plus the number of listings.
4. Adds date features (weekday, month, hour) and a venue key, and saves `data/shows.csv`.

**Where it fits:** `01_build_music_db.py` builds the database, then **this notebook** (to-do step 1), then `04_build_artists.ipynb` and `05_build_venues.ipynb` look up the artists and venues. The model table (step 7) joins `shows` to `artists` on `artist_key` and to `venues` on `venue_key`.

---

**Input:** `data/music.db` (table `gametime`)
**Output:** `data/shows.csv`, one row per show (`event_id`)

**Decisions made here (change them in the setup cell):**
- **Price timing:** `TARGET_DAYS = 14` days before the show, using the nearest snapshot within `WINDOW_DAYS = 4` days either side.
- **Target:** both `price_min` (cheapest ticket) and `price_median` (typical listing) are saved, with log versions. Pick one for the model after looking at the comparison in section 5.
- **US only:** Gametime has no state or country column, so US shows **cannot be filtered here**. `05_build_venues.ipynb` finds each venue's country (`is_us`); drop non-US shows when joining in the model step.
- **Local time:** `eventStartAt` is in UTC with no time zone, so weekday and hour use a rough local time (UTC minus 6 hours). This is right for most evening shows but can put a late show on the wrong day. `05_build_venues.ipynb` saves each venue's time zone if we want to fix it later.

## 1. Setup

In [ ]:
import re
import sqlite3
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

DB_PATH = Path("data/music.db")
OUT_PATH = Path("data/shows.csv")

TARGET_DAYS = 14          # price this many days before the show
WINDOW_DAYS = 4           # accept a snapshot this many days either side of TARGET_DAYS
DROP_THEME_NIGHTS = True  # tribute / party nights have no single artist to look up
LOCAL_OFFSET_HOURS = -6   # rough UTC -> local conversion (see note above)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

## 2. Load one row per show
Two queries on the database: one row of facts per show, and the snapshots near the target day.

In [ ]:
conn = sqlite3.connect(DB_PATH)

# One row of facts per show (all days)
shows = pd.read_sql("""
    SELECT eventId AS event_id,
           MIN(eventName)   AS name,
           MIN(venueName)   AS venue,
           MIN(venueCity)   AS city,
           MIN(eventStartAt) AS start_utc,
           MAX(listings)    AS max_listings,
           SUM(dayState = 'observed' AND minPrice IS NOT NULL) AS n_priced_days
    FROM gametime
    GROUP BY eventId""", conn)
print(f"{len(shows):,} shows in the database")

# Priced snapshots near the target day
lo, hi = TARGET_DAYS - WINDOW_DAYS, TARGET_DAYS + WINDOW_DAYS
snaps = pd.read_sql(f"""
    SELECT eventId AS event_id, date AS snapshot_date, daysToEvent AS days_to_show,
           listings, minPrice AS price_min, medianPrice AS price_median
    FROM gametime
    WHERE dayState = 'observed' AND minPrice IS NOT NULL AND medianPrice IS NOT NULL
      AND daysToEvent BETWEEN {lo} AND {hi}""", conn)
conn.close()
print(f"{len(snaps):,} priced snapshots between {lo} and {hi} days before the show")

## 3. Clean the show names into artist names
This is the same cleaning (and the same `artist_key` recipe) as `04_build_artists.ipynb`, so the two tables join. If you change one, change the other.

In [ ]:
# Festival passes / multi-day tickets are not one artist playing one night
PASS = re.compile(r"\b(?:\d+[- ]day|day pass|weekend pass|season pass|festival|fest|"
                  r"(?:single|multi)[- ]day|(?:general admission|ga|vip) pass|pass)\b", re.I)
# Tribute / party nights have no single artist (kept narrow so "Night Ranger" survives)
THEME = re.compile(r"tribute|dance party|\bparty\b|brunch|happy hour|karaoke|open mic|jam session|"
                   r"showcase|celebration of|emo nite|drag |bingo|comedy", re.I)
# Show info that is not part of the artist name
NOISE = re.compile(r"\((?:rescheduled|moved|postponed|new date|new venue|late show|early show|"
                   r"18\+|21\+|all ages|sold out)[^)]*\)|\*[^*]*\*|\[[^\]]*\]|"
                   r"\b(?:rescheduled|moved|postponed) from .*$|\b(?:18|21)\+(?: event| show)?", re.I)
SEP = re.compile(
    r"\s+(?:w/|with|feat\.?|featuring|ft\.?|x|vs\.?|\+|\|)\s+"   # 'X w/ Y'
    r"|,|;"                                                         # lists
    r"|\s/\s?|\s?/\s"                                               # ' / ' (keeps AC/DC)
    r"|\s+[-–—]\s+"                                       # ' - Tour Name'
    r"|:\s", re.I)
TRAILING = re.compile(r"\s+(?:album release|record release|ep release|release show|residency.*|live|"
                      r"the .*tour|.*\btour\b.*|\d{4})$", re.I)
ENSEMBLE = re.compile(r"\s+(trio|quartet|quintet|sextet|septet|octet|nonet|big band|orchestra|band|ensemble)$", re.I)


def parse_headliner(name):
    if not isinstance(name, str):
        return None
    s = NOISE.sub(" ", name)
    s = re.sub(r"\([^)]*\)", " ", s).replace("�", "'").strip()
    m = re.search(r"\b(presents?):?\s+(.+)", s, re.I)
    if m:
        before, word, after = s[:m.start()].strip(), m.group(1).lower(), m.group(2).strip()
        artist_first = word == "present" or after.startswith(('"', "'")) or "tour" in after.lower()
        s = before if (artist_first and before) else after
    first = SEP.split(s, maxsplit=1)[0]
    first = TRAILING.sub("", first).strip(" -:,.'\"")
    return re.sub(r"\s{2,}", " ", first) or None


def make_key(name):
    """Join key: no accents, lowercase, letters and digits only, no leading 'the'."""
    if not isinstance(name, str) or not name.strip():
        return None
    s = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    s = re.sub(r"[^a-z0-9]+", " ", s.lower().replace("&", " and ")).strip()
    return re.sub(r"^the\s+", "", s) or None


shows["is_pass"] = shows["name"].str.contains(PASS, na=False)
shows["is_theme_night"] = shows["name"].str.contains(THEME, na=False)
shows["artist_name"] = shows["name"].map(parse_headliner)
shows["artist_key"] = shows["artist_name"].map(make_key)
# venue_key joins to venues.csv: venue name + city, cleaned the same way
shows["venue_key"] = shows["venue"].map(make_key).fillna("") + "|" + shows["city"].map(make_key).fillna("")

shows[["name", "artist_name", "artist_key"]].sample(8, random_state=3)

## 4. Keep the right shows and attach the price
Each show keeps the snapshot **closest to the target day** (ties go to the later snapshot). Shows with no snapshot in the window are dropped: they were added to Gametime too late, or are too far out for our snapshots (the last snapshot is Oct 21).

In [ ]:
funnel = [("shows in database", len(shows))]

keep = shows[~shows["is_pass"]]
funnel.append(("minus festival / multi-day passes", len(keep)))
if DROP_THEME_NIGHTS:
    keep = keep[~keep["is_theme_night"]]
    funnel.append(("minus tribute / party nights", len(keep)))
keep = keep[keep["artist_key"].notna()]
funnel.append(("minus no artist name", len(keep)))

# the snapshot closest to TARGET_DAYS for each show
snaps["gap"] = (snaps["days_to_show"] - TARGET_DAYS).abs()
best = (snaps.sort_values(["event_id", "gap", "days_to_show"])
             .drop_duplicates("event_id")
             .drop(columns="gap"))
keep = keep.merge(best, on="event_id", how="inner")
funnel.append((f"with a price {TARGET_DAYS} +/- {WINDOW_DAYS} days out", len(keep)))

pd.DataFrame(funnel, columns=["step", "shows"]).assign(dropped=lambda d: -d["shows"].diff())

## 5. Add features, check, and save

In [ ]:
out = keep.copy()
out["start_utc"] = pd.to_datetime(out["start_utc"])
local = out["start_utc"] + pd.Timedelta(hours=LOCAL_OFFSET_HOURS)   # rough local start time
out["start_local_approx"] = local
out["weekday"] = local.dt.day_name()
out["is_weekend"] = local.dt.dayofweek.isin([4, 5])                  # Friday and Saturday
out["month"] = local.dt.month
out["show_hour"] = local.dt.hour
out["log_price_min"] = np.log(out["price_min"])
out["log_price_median"] = np.log(out["price_median"])

cols = ["event_id", "name", "artist_key", "artist_name", "venue", "city", "venue_key",
        "start_utc", "start_local_approx", "weekday", "is_weekend", "month", "show_hour",
        "snapshot_date", "days_to_show", "listings", "max_listings", "n_priced_days",
        "price_min", "price_median", "log_price_min", "log_price_median"]
out = out[cols].sort_values("start_utc").reset_index(drop=True)
out.to_csv(OUT_PATH, index=False)

print(f"Saved {OUT_PATH}: {len(out):,} shows | {out['artist_key'].nunique():,} artists | "
      f"{out['venue_key'].nunique():,} venues")
print(f"  show dates: {out['start_utc'].min():%Y-%m-%d} to {out['start_utc'].max():%Y-%m-%d}")
dupes = out.assign(day=out["start_local_approx"].dt.date).duplicated(["artist_key", "venue_key", "day"]).sum()
print(f"  possible duplicates (same artist, venue and day): {dupes:,}")
out.head()

### Which target? `price_min` vs `price_median`
Both are right-skewed, so we model the log. Compare them before choosing (the choice is on the to-do list).

In [ ]:
summary = out[["price_min", "price_median", "log_price_min", "log_price_median", "listings"]].describe().T
summary["skew"] = out[summary.index].skew()
summary

In [ ]:
# How many shows fall in each month, and how many sit near the edges of the window?
print(out["start_utc"].dt.to_period("M").value_counts().sort_index().to_string())
print()
print("days before show of the snapshot used:")
print(out["days_to_show"].value_counts().sort_index().to_string())

In [ ]:
# Busiest venues and artists (what will dominate the model)
print(out["venue"].value_counts().head(10).to_string())
print()
print(out["artist_name"].value_counts().head(10).to_string())